In [4]:
"""
Trialwise semantic embeddings + chosen-choice sentiment, with incremental
per-subject NPZ caching.

Design
------
- Embedding is done once across ALL subjects: every unique prompt is embedded
  a single time and cached in a master NPZ (prompt -> vector). Re-runs reload
  the cache and only embed prompts not already seen.
- Per-subject NPZs store one array per representation. On re-run, existing reps
  are kept and only MISSING reps are computed and appended.
- If the embedding/sentiment model in an existing NPZ differs from the requested
  model, we raise (no silent recompute, no silent mixing).
"""

from shared.main import *
from shared.nlp import get_sentence_embeddings
from collections import defaultdict

# ============================================================
# Config
# ============================================================

CORE_REPS = [
    "choice",
    "choice_diff",
    "choice_diff_local",
    "choice_diff_prev1",
    "choice_diff_prev3",
    "choice_diff_history",
    "running_mean",
    "running_variance",
    "change",
]

EMBEDDING_MODEL_NAME = "openai"
SENTIMENT_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"
HISTORY_REPS = {"running_mean", "running_variance", "change"}

# ============================================================
# Small helpers
# ============================================================

def clean_text(x):
    if pd.isna(x):
        return "none"
    x = " ".join(str(x).replace("\n", " ").split()).strip()
    return x if x else "none"

def l2_normalize(X, axis=1, eps=1e-8):
    X = np.asarray(X, dtype=float)
    return X / (np.linalg.norm(X, axis=axis, keepdims=True) + eps)

def _validate_rep_names(rep_names):
    allowed = {
        "choice", "choice_diff", "choice_diff_local", "choice_diff_history",
        "running_mean", "running_variance", "change",
    }
    rep_names = list(rep_names)
    for rep in rep_names:
        if rep in allowed:
            continue
        if rep.startswith("choice_diff_prev") and rep[len("choice_diff_prev"):].isdigit():
            continue
        raise ValueError(f"Unknown representation: {rep}")
    return rep_names

def _prompt_cols_for_rep(rep):
    """(chosen_col, unchosen_col) for a prompt-based rep, else (None, None)."""
    if rep == "choice":
        return "_choice_chosen_prompt", None
    if rep == "choice_diff":
        return "_choice_diff_chosen_prompt", "_choice_diff_unchosen_prompt"
    if rep == "choice_diff_local":
        return "_choice_diff_local_chosen_prompt", "_choice_diff_local_unchosen_prompt"
    if rep.startswith("choice_diff_prev"):
        n = int(rep[len("choice_diff_prev"):])
        return f"_choice_diff_prev{n}_chosen_prompt", f"_choice_diff_prev{n}_unchosen_prompt"
    if rep == "choice_diff_history":
        return "_choice_diff_history_chosen_prompt", "_choice_diff_history_unchosen_prompt"
    return None, None

# ============================================================
# Prompt/text construction
# ============================================================

def build_core_prompt_df(
    subject_data,
    *,
    rep_names=CORE_REPS,
    trial_col=None,
    character_col="character_role_num",
    prevslide_col="prevslide_text",
    chosen_col="chosen_text",
    unchosen_col="unchosen_text",
):
    """Build one stacked trial-level prompt dataframe from subject_data."""
    rep_names = _validate_rep_names(rep_names)

    rows, key_lookup = [], {}

    for sub_key, sd in subject_data.items():
        if sd is None or sd.get("behavior") is None:
            continue

        sub_key_str = str(sub_key)
        key_lookup[sub_key_str] = sub_key

        beh = sd["behavior"].copy().reset_index(drop=True)

        role_col = character_col
        if role_col not in beh.columns:
            if "char_role_num" in beh.columns:
                role_col = "char_role_num"
            else:
                raise KeyError(
                    f"Subject {sub_key} has no character role column "
                    f"({character_col!r} or 'char_role_num')."
                )

        needed = [role_col, prevslide_col, chosen_col, unchosen_col]
        missing = [c for c in needed if c not in beh.columns]
        if missing:
            raise KeyError(f"Subject {sub_key} missing behavior columns: {missing}")

        cols = needed.copy()
        if trial_col is not None:
            if trial_col not in beh.columns:
                raise KeyError(f"Subject {sub_key} missing trial_col={trial_col!r}")
            cols.append(trial_col)

        d = beh[cols].rename(columns={
            role_col: "character_role_num",
            prevslide_col: "prevslide_text",
            chosen_col: "chosen_text",
            unchosen_col: "unchosen_text",
        })

        d["_sub_key"] = sub_key_str
        d["sub_id"] = sd.get("sub_id", sub_key_str)
        d["_trial_order"] = (
            np.arange(len(d)) if trial_col is None else d[trial_col].to_numpy()
        )

        for c in ["prevslide_text", "chosen_text", "unchosen_text"]:
            d[c] = d[c].map(clean_text)

        rows.append(d)

    if not rows:
        raise ValueError("No usable behavior found in subject_data.")

    out = pd.concat(rows, ignore_index=True)
    out = out.sort_values(["_sub_key", "_trial_order"]).reset_index(drop=True)

    # No-context base prompts
    out["_choice_chosen_prompt"] = out["chosen_text"]
    out["_choice_unchosen_prompt"] = out["unchosen_text"]
    out["_choice_diff_chosen_prompt"] = out["chosen_text"]
    out["_choice_diff_unchosen_prompt"] = out["unchosen_text"]

    def make_context_prompt(row, option_text, history_col=None):
        pieces = [f"Current scene: {clean_text(row['prevslide_text'])}"]
        if history_col is not None:
            pieces.append(
                "Previous choices with this character: "
                f"{clean_text(row[history_col])}"
            )
        pieces.append(f"Current option: {clean_text(option_text)}")
        return "\n".join(pieces)

    def add_history_column(history_k, col_name):
        history = defaultdict(list)
        hist_text = {}
        for idx, row in out.sort_values(["_sub_key", "_trial_order"]).iterrows():
            key = (row["_sub_key"], row["character_role_num"])
            prior = history[key]
            use_prior = prior if history_k is None else prior[-int(history_k):]
            if not use_prior:
                hist_text[idx] = "none"
            else:
                hist_text[idx] = " | ".join(
                    f"{i + 1}. {txt}" for i, txt in enumerate(use_prior)
                )
            history[key].append(clean_text(row["chosen_text"]))  # after prompt built
        out[col_name] = out.index.map(hist_text)

    if "choice_diff_local" in rep_names:
        out["_choice_diff_local_chosen_prompt"] = out.apply(
            lambda r: make_context_prompt(r, r["chosen_text"]), axis=1)
        out["_choice_diff_local_unchosen_prompt"] = out.apply(
            lambda r: make_context_prompt(r, r["unchosen_text"]), axis=1)

    prev_ns = sorted({
        int(rep[len("choice_diff_prev"):])
        for rep in rep_names if rep.startswith("choice_diff_prev")
    })
    for n_back in prev_ns:
        hist_col = f"_same_character_history_prev{n_back}"
        add_history_column(history_k=n_back, col_name=hist_col)
        out[f"_choice_diff_prev{n_back}_chosen_prompt"] = out.apply(
            lambda r, hc=hist_col: make_context_prompt(r, r["chosen_text"], hc), axis=1)
        out[f"_choice_diff_prev{n_back}_unchosen_prompt"] = out.apply(
            lambda r, hc=hist_col: make_context_prompt(r, r["unchosen_text"], hc), axis=1)

    if "choice_diff_history" in rep_names:
        hist_col = "_same_character_history_all"
        add_history_column(history_k=None, col_name=hist_col)
        out["_choice_diff_history_chosen_prompt"] = out.apply(
            lambda r: make_context_prompt(r, r["chosen_text"], hist_col), axis=1)
        out["_choice_diff_history_unchosen_prompt"] = out.apply(
            lambda r: make_context_prompt(r, r["unchosen_text"], hist_col), axis=1)

    return out, key_lookup

# ============================================================
# Embeddings
# ============================================================

def embed_unique_prompts(
    prompts,
    *,
    model_name=EMBEDDING_MODEL_NAME,
    batch_size=128,
    cache_path=None,
):
    """
    Embed a list of prompts, returning {clean_prompt: vector}.

    If cache_path is given, previously embedded prompts are loaded from it and
    only new prompts are sent to the embedding helper; the cache is then updated.
    All cached vectors must share the same model_name or we raise.
    """
    prompts = [clean_text(p) for p in prompts]
    unique = pd.Series(prompts).drop_duplicates().tolist()

    emb_map, cached_dim = {}, None
    if cache_path is not None and Path(cache_path).exists():
        emb_map, cached_dim = _load_embedding_cache(cache_path, model_name)

    todo = [p for p in unique if p not in emb_map]
    if todo:
        E = _embed_batched(todo, model_name=model_name, batch_size=batch_size)
        if cached_dim is not None and E.shape[1] != cached_dim:
            raise ValueError(
                f"New embeddings have dim {E.shape[1]} but cache has {cached_dim}."
            )
        emb_map.update(zip(todo, E))
        if cache_path is not None:
            _save_embedding_cache(cache_path, emb_map, model_name)

    return emb_map

def _embed_batched(texts, *, model_name, batch_size):
    """The one and only batched wrapper around get_sentence_embeddings."""
    texts = [clean_text(x) for x in texts]
    if not texts:
        raise ValueError("No texts to embed.")

    chunks = []
    for start in tqdm(range(0, len(texts), batch_size),
                      desc=f"Embedding ({model_name})"):
        batch = texts[start:start + batch_size]
        E_batch = np.asarray(
            get_sentence_embeddings(batch, model=model_name, normalize=True),  # noqa: F821
            dtype=np.float32,
        )

        if E_batch.ndim == 1:
            if len(batch) == 1 and E_batch.size > 0:
                E_batch = E_batch[None, :]
            else:
                raise ValueError(
                    f"get_sentence_embeddings(model={model_name!r}) returned a 1D "
                    f"array {E_batch.shape} for batch size {len(batch)} "
                    f"(start={start}); likely an API error returning np.array([])."
                )
        if E_batch.ndim != 2:
            raise ValueError(f"Expected 2D embeddings, got {E_batch.shape}.")
        if E_batch.shape[0] != len(batch):
            raise ValueError(
                f"Got {E_batch.shape[0]} embeddings for {len(batch)} texts "
                f"(start={start})."
            )
        chunks.append(E_batch)

    E = np.vstack(chunks).astype(np.float32)
    return l2_normalize(E).astype(np.float32)

def _load_embedding_cache(cache_path, model_name):
    z = np.load(cache_path, allow_pickle=True)
    stored_model = str(z["model"])
    if stored_model != model_name:
        raise ValueError(
            f"Embedding cache {cache_path} was built with model {stored_model!r}, "
            f"but {model_name!r} was requested. Use a different cache_path."
        )
    prompts = z["prompts"].astype(str)
    vectors = z["vectors"].astype(np.float32)
    emb_map = {p: v for p, v in zip(prompts, vectors)}
    return emb_map, (vectors.shape[1] if len(vectors) else None)

def _save_embedding_cache(cache_path, emb_map, model_name):
    Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
    prompts = np.asarray(list(emb_map.keys()), dtype=str)
    vectors = np.vstack(list(emb_map.values())).astype(np.float32)
    np.savez_compressed(
        cache_path, prompts=prompts, vectors=vectors, model=np.array(model_name)
    )

def compute_trial_vectors(
    prompt_df,
    emb_map,
    rep_names,
    *,
    variance_min_prior=2,
):
    """
    Turn an embedding map + prompt_df into {rep: n_trials x dim array}.

    Pure function over already-computed embeddings; no API calls. Returns arrays
    aligned to prompt_df's row order. running_variance is returned as (n, 1).
    """
    rep_names = _validate_rep_names(rep_names)
    needs_history = any(r in HISTORY_REPS for r in rep_names)
    prompt_reps = [r for r in rep_names if r not in HISTORY_REPS]
    if needs_history and "choice_diff" not in prompt_reps:
        prompt_reps = ["choice_diff"] + prompt_reps

    def lookup(col):
        return np.vstack([emb_map[clean_text(x)] for x in prompt_df[col]]).astype(np.float32)

    trial_vectors = {}
    for rep in prompt_reps:
        chosen_col, unchosen_col = _prompt_cols_for_rep(rep)
        E_chosen = lookup(chosen_col)
        if rep == "choice":
            trial_vectors[rep] = E_chosen
        else:
            E_unchosen = lookup(unchosen_col)
            trial_vectors[rep] = l2_normalize(E_chosen - E_unchosen).astype(np.float32)

    if needs_history:
        base = l2_normalize(trial_vectors["choice_diff"]).astype(np.float32)
        n_trials, dim = base.shape
        running_mean = np.zeros((n_trials, dim), dtype=np.float32)
        running_variance = np.full(n_trials, np.nan, dtype=np.float32)
        change = np.zeros((n_trials, dim), dtype=np.float32)

        for _, g in prompt_df.groupby(["_sub_key", "character_role_num"], sort=False):
            idx = g.sort_values("_trial_order").index.to_numpy()
            Vg = base[idx]
            for k, trial_idx in enumerate(idx):
                mu = np.zeros(dim, dtype=np.float32) if k == 0 \
                    else Vg[:k].mean(axis=0).astype(np.float32)
                if k > 0:
                    running_mean[trial_idx] = mu
                if k >= variance_min_prior:
                    running_variance[trial_idx] = np.float32(
                        1.0 - np.linalg.norm(Vg[:k].mean(axis=0))
                    )
                change[trial_idx] = (Vg[k] - mu).astype(np.float32)

        if "running_mean" in rep_names:
            trial_vectors["running_mean"] = running_mean
        if "running_variance" in rep_names:
            trial_vectors["running_variance"] = running_variance[:, None]
        if "change" in rep_names:
            trial_vectors["change"] = change

    return {rep: trial_vectors[rep] for rep in rep_names}

# ============================================================
# Sentiment
# ============================================================

def _get_roberta_sentiment_model(model_name=SENTIMENT_MODEL_NAME):
    if "_choice_sentiment_model" not in globals():
        from transformers import pipeline
        globals()["_choice_sentiment_model"] = pipeline(
            "sentiment-analysis", model=model_name, return_all_scores=True,
        )
    return globals()["_choice_sentiment_model"]

def calculate_roberta_compound(positive, negative, neutral):
    total = positive + negative + neutral
    if total <= 0:
        return np.nan
    p, n, z = positive / total, negative / total, neutral / total
    return np.round((p - n) * (1 - z), 3)

def estimate_choice_sentiment(texts, *, batch_size=128, model_name=SENTIMENT_MODEL_NAME):
    """One row per input text (input order preserved); dedups internally."""
    texts = [clean_text(x) for x in texts]
    unique = pd.Series(texts).drop_duplicates().tolist()
    model = _get_roberta_sentiment_model(model_name=model_name)
    label_map = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}

    score_map = {}
    for start in tqdm(range(0, len(unique), batch_size), desc="Scoring sentiment"):
        batch = unique[start:start + batch_size]
        for text, scores in zip(batch, model(batch)):
            row = {"negative": 0.0, "neutral": 0.0, "positive": 0.0}
            for item in scores:
                lab = label_map.get(item["label"])
                if lab is not None:
                    row[lab] = np.round(float(item["score"]), 3)
            row["compound"] = calculate_roberta_compound(
                row["positive"], row["negative"], row["neutral"])
            score_map[text] = row

    return pd.DataFrame([score_map[t] for t in texts])

# ============================================================
# Incremental per-subject NPZ save
# ============================================================

def _subject_npz_path(out_dir, sub_id):
    sub_label = sub_id if sub_id.startswith("sub-") else f"sub-{sub_id}"
    return Path(out_dir) / f"{sub_label}_choice_features.npz"

def _read_existing_npz(path, *, embedding_model, sentiment_model):
    """Load an existing subject NPZ as a mutable dict, enforcing model match."""
    z = np.load(path, allow_pickle=True)
    store = {k: z[k] for k in z.files}
    have_emb = str(store.get("embedding_model", ""))
    have_sent = str(store.get("sentiment_model", ""))
    if have_emb and have_emb != embedding_model:
        raise ValueError(
            f"{path}: existing embedding_model {have_emb!r} != requested "
            f"{embedding_model!r}. Refusing to mix models."
        )
    if have_sent and have_sent != sentiment_model:
        raise ValueError(
            f"{path}: existing sentiment_model {have_sent!r} != requested "
            f"{sentiment_model!r}. Refusing to mix models."
        )
    return store

def save_subject_choice_feature_npzs(
    subject_data,
    out_dir,
    *,
    rep_names=CORE_REPS,
    embedding_model_name=EMBEDDING_MODEL_NAME,
    sentiment_model_name=SENTIMENT_MODEL_NAME,
    embedding_batch_size=128,
    sentiment_batch_size=128,
    variance_min_prior=2,
    embedding_cache_name="_embedding_cache.npz",
):
    """
    Save/extend one NPZ per subject with trialwise embeddings + chosen sentiment.

    Incremental behaviour:
      - Each unique prompt is embedded once across all subjects; vectors are
        cached in out_dir/embedding_cache_name and reused on later runs.
      - For each subject, existing reps are kept; only reps not already present
        (and sentiment, if absent) are computed and merged in.
      - If a subject is already complete for the requested reps, it is skipped
        without any embedding/sentiment work for it.
      - Model mismatch between request and an existing NPZ raises.
    """
    rep_names = _validate_rep_names(rep_names)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cache_path = out_dir / embedding_cache_name

    prompt_df, key_lookup = build_core_prompt_df(subject_data, rep_names=rep_names)

    # Decide per subject what is missing, so we only do work that's needed.
    plan = {}  # sub_key_str -> {"store": dict|None, "missing_reps": [...], "need_sent": bool}
    for sub_key_str, g in prompt_df.groupby("_sub_key", sort=False):
        sub_id = str(g["sub_id"].iloc[0])
        path = _subject_npz_path(out_dir, sub_id)
        if path.exists():
            store = _read_existing_npz(
                path, embedding_model=embedding_model_name,
                sentiment_model=sentiment_model_name)
            missing = [r for r in rep_names if f"embedding_{r}" not in store]
            need_sent = "sentiment_compound" not in store
        else:
            store, missing, need_sent = None, list(rep_names), True
        plan[sub_key_str] = {"store": store, "missing": missing, "need_sent": need_sent}

    # Which reps need computing anywhere? (history reps pull in choice_diff)
    reps_to_compute = sorted(
        {r for p in plan.values() for r in p["missing"]},
        key=rep_names.index,
    )

    summary_rows = []

    # --- Embeddings + trial vectors only if some rep is missing somewhere ---
    trial_vectors = {}
    if reps_to_compute:
        needs_history = any(r in HISTORY_REPS for r in reps_to_compute)
        prompt_reps = [r for r in reps_to_compute if r not in HISTORY_REPS]
        if needs_history and "choice_diff" not in prompt_reps:
            prompt_reps = ["choice_diff"] + prompt_reps

        prompt_cols = []
        for rep in prompt_reps:
            c1, c2 = _prompt_cols_for_rep(rep)
            prompt_cols += [c for c in (c1, c2) if c is not None]
        prompt_cols = list(dict.fromkeys(prompt_cols))

        all_prompts = [p for col in prompt_cols for p in prompt_df[col].tolist()]
        emb_map = embed_unique_prompts(
            all_prompts, model_name=embedding_model_name,
            batch_size=embedding_batch_size, cache_path=cache_path)

        # compute_trial_vectors needs the union (history may add choice_diff)
        compute_reps = list(dict.fromkeys(reps_to_compute + (
            ["choice_diff"] if needs_history else [])))
        trial_vectors = compute_trial_vectors(
            prompt_df, emb_map, compute_reps, variance_min_prior=variance_min_prior)

    # --- Sentiment only for subjects that need it ---
    need_sent_keys = {k for k, p in plan.items() if p["need_sent"]}
    sentiment = None
    if need_sent_keys:
        mask = prompt_df["_sub_key"].isin(need_sent_keys)
        sent_vals = estimate_choice_sentiment(
            prompt_df.loc[mask, "chosen_text"].tolist(),
            batch_size=sentiment_batch_size, model_name=sentiment_model_name)
        sentiment = pd.DataFrame(
            sent_vals.to_numpy(), index=prompt_df.loc[mask].index,
            columns=sent_vals.columns)

    # --- Merge + write per subject ---
    for sub_key_str, g in prompt_df.groupby("_sub_key", sort=False):
        info = plan[sub_key_str]
        sub_id = str(g["sub_id"].iloc[0])
        path = _subject_npz_path(out_dir, sub_id)
        idx = g.index.to_numpy()

        if not info["missing"] and not info["need_sent"]:
            summary_rows.append({
                "sub_key": sub_key_str, "sub_id": sub_id, "n_trials": len(g),
                "added_reps": [], "added_sentiment": False,
                "status": "skipped_complete", "out_path": str(path)})
            continue

        store = dict(info["store"]) if info["store"] is not None else {}

        # Fixed identity / metadata
        store.update({
            "sub_key": np.array(sub_key_str),
            "sub_id": np.array(sub_id),
            "embedding_model": np.array(embedding_model_name),
            "sentiment_model": np.array(sentiment_model_name),
            "trial_order": g["_trial_order"].to_numpy(),
            "character_role_num": g["character_role_num"].to_numpy(),
            "chosen_text": g["chosen_text"].to_numpy(dtype=str),
            "unchosen_text": g["unchosen_text"].to_numpy(dtype=str),
            "prevslide_text": g["prevslide_text"].to_numpy(dtype=str),
        })

        for rep in info["missing"]:
            arr = np.asarray(trial_vectors[rep][idx], dtype=np.float32)
            if arr.ndim == 1:
                arr = arr[:, None]
            if arr.shape[0] != len(g):
                raise ValueError(
                    f"{sub_id} rep={rep}: rows {arr.shape[0]} != trials {len(g)}")
            store[f"embedding_{rep}"] = arr

        if info["need_sent"] and sentiment is not None:
            for col in ["positive", "negative", "neutral", "compound"]:
                store[f"sentiment_{col}"] = (
                    sentiment.loc[idx, col].to_numpy(np.float32))

        # rep_names reflects everything now stored
        stored_reps = [r for r in CORE_REPS if f"embedding_{r}" in store]
        store["rep_names"] = np.asarray(stored_reps, dtype=str)

        np.savez_compressed(path, **store)
        summary_rows.append({
            "sub_key": sub_key_str, "sub_id": sub_id, "n_trials": len(g),
            "added_reps": info["missing"],
            "added_sentiment": bool(info["need_sent"]),
            "status": "created" if info["store"] is None else "extended",
            "out_path": str(path)})

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(out_dir / "choice_feature_npz_manifest.csv", index=False)
    return summary


# ============================================================
# Optional: load features back from a subject NPZ
# ============================================================

def load_subject_choice_features(path):
    """Return (embeddings dict, sentiment DataFrame, meta dict) from a subject NPZ."""
    z = np.load(path, allow_pickle=True)
    reps = z["rep_names"].astype(str).tolist()
    embeddings = {r: z[f"embedding_{r}"] for r in reps if f"embedding_{r}" in z.files}
    sent_cols = ["positive", "negative", "neutral", "compound"]
    sentiment = pd.DataFrame({
        c: z[f"sentiment_{c}"] for c in sent_cols if f"sentiment_{c}" in z.files
    }) if any(f"sentiment_{c}" in z.files for c in sent_cols) else None
    meta = {
        "sub_id": str(z["sub_id"]),
        "embedding_model": str(z["embedding_model"]),
        "sentiment_model": str(z["sentiment_model"]),
        "rep_names": reps,
    }
    return embeddings, sentiment, meta


In [5]:
# ============================================================
# Run: load behavior only, then save per-subject NLP feature NPZs
# ============================================================

feature_dir = PROJECT_ROOT / "data" / "narratives" / "choice_features"
os.makedirs(feature_dir, exist_ok=True)

subject_data = {}

for sub_id in tqdm(incl_subs, desc="Loading behavior"):
    behavior = load_behavior(sub_id)

    if behavior is None or len(behavior) == 0:
        print(f"[Skipping] {sub_id}: no behavior found")
        continue

    subject_data[sub_id] = {
        "sub_id": str(sub_id),
        "behavior": behavior.reset_index(drop=True),
    }

feature_summary = save_subject_choice_feature_npzs(
    subject_data,
    feature_dir,
    rep_names=CORE_REPS,
    embedding_model_name=EMBEDDING_MODEL_NAME,
    sentiment_model_name=SENTIMENT_MODEL_NAME,
    embedding_batch_size=128,
    sentiment_batch_size=128,
    variance_min_prior=2,
)

print(f"Saved per-subject NLP feature NPZs to:\n{feature_dir}")
feature_summary


Embedding (openai): 100%|██████████| 45/45 [00:33<00:00,  1.36it/s]
Device set to use mps:0
Scoring sentiment: 100%|██████████| 1/1 [00:04<00:00,  4.09s/it]


Saved per-subject NLP feature NPZs to:
/Users/matty_gee/Desktop/Social/SocialCUD/data/narratives/choice_features


,sub_key,sub_id,n_trials,added_reps,added_sentiment,status,out_path
0,18001,18001,63,"[choice, choice_diff, choice_diff_local, choic...",True,created,/Users/matty_gee/Desktop/Social/SocialCUD/data...
1,18002,18002,63,"[choice, choice_diff, choice_diff_local, choic...",True,created,/Users/matty_gee/Desktop/Social/SocialCUD/data...
2,18003,18003,63,"[choice, choice_diff, choice_diff_local, choic...",True,created,/Users/matty_gee/Desktop/Social/SocialCUD/data...
3,18004,18004,63,"[choice, choice_diff, choice_diff_local, choic...",True,created,/Users/matty_gee/Desktop/Social/SocialCUD/data...
4,18005,18005,63,"[choice, choice_diff, choice_diff_local, choic...",True,created,/Users/matty_gee/Desktop/Social/SocialCUD/data...
...,...,...,...,...,...,...,...
74,23010,23010,63,"[choice, choice_diff, choice_diff_local, choic...",True,created,/Users/matty_gee/Desktop/Social/SocialCUD/data...
75,23015,23015,63,"[choice, choice_diff, choice_diff_local, choic...",True,created,/Users/matty_gee/Desktop/Social/SocialCUD/data...
76,23019,23019,63,"[choice, choice_diff, choice_diff_local, choic...",True,created,/Users/matty_gee/Desktop/Social/SocialCUD/data...
77,23020,23020,63,"[choice, choice_diff, choice_diff_local, choic...",True,created,/Users/matty_gee/Desktop/Social/SocialCUD/data...
